# Medical Report Simplifier
This notebook demonstrates an NLP pipeline that takes a medical report (image or PDF), extracts text using OCR, identifies medical terminologies using BioBERT, and uses an LLM to generate patient-friendly explanations and advice.

## 1. Installation and Imports
Install necessary packages. Note: You need to have Tesseract OCR installed on your system for `pytesseract` to work.

In [1]:
!pip install pytesseract pdf2image transformers torch scikit-learn google-generativeai python-dotenv pillow
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.1/en_core_sci_sm-0.5.1.tar.gz

  Using cached https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.1/en_core_sci_sm-0.5.1.tar.gz (15.9 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'


In [2]:
import os
import re
import torch
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
from transformers import pipeline, AutoTokenizer, AutoModel
from dotenv import load_dotenv
import google.generativeai as genai

# Load environment variables from .env file
# Make sure to create a .env file in the same directory and add your GEMINI_API_KEY
load_dotenv()

# Configure Gemini API (Using Gemini as the LLM)
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    print("WARNING: GEMINI_API_KEY not found in environment or .env file.")
else:
    genai.configure(api_key=api_key)
    # We will use the gemini-2.5-flash model
    generation_config = {
        "temperature": 0.3,
        "top_p": 0.95,
        "top_k": 40,
        "max_output_tokens": 8192,
        "response_mime_type": "text/plain",
    }
    model = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        generation_config=generation_config,
    )
    print("API Key loaded successfully.")

API Key loaded successfully.


In [3]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

## 2. OCR function for Extraction
Extract text from image or PDF.

In [4]:
def extract_text(file_path):
    text = ""
    if file_path.lower().endswith('.pdf'):
        # Convert PDF pages to images
        pages = convert_from_path(file_path)
        for page in pages:
            text += pytesseract.image_to_string(page) + "\n"
    elif file_path.lower().endswith(('.png', '.jpg', '.jpeg', '.tiff', '.bmp')):
        # Process image directly
        image = Image.open(file_path)
        text = pytesseract.image_to_string(image)
    else:
        raise ValueError("Unsupported file format. Please provide a PDF or image file.")
    return text

# Example usage (Uncomment and replace with your file path):
# raw_text = extract_text("sample_medical_report.pdf")
# print(raw_text)

## 3. Medical Term Extraction & BioBERT Embeddings
Use HuggingFace pipelines (`d4data/biomedical-ner-all`) to extract medical terminology. Then, extract embeddings via BioBERT (`dmis-lab/biobert-v1.1`).

In [5]:
print("Loading BioBERT model and tokenizer...")
model_name = "dmis-lab/biobert-v1.1"
tokenizer = AutoTokenizer.from_pretrained(model_name)
biobert_model = AutoModel.from_pretrained(model_name)

# Using a standard NER pipeline for biomedical entity extraction.
print("Loading NER Pipeline...")
ner_pipeline = pipeline("ner", model="d4data/biomedical-ner-all", tokenizer="d4data/biomedical-ner-all", aggregation_strategy="simple")

def extract_and_embed_medical_terms(text):
    print("Extracting medical terms from text...")
    entities = ner_pipeline(text)
    
    # Keep it simple for this example: extract all unique entities found that are longer than 2 characters
    terms = list(set([ent['word'] for ent in entities if len(ent['word']) > 2]))
    print(f"Found {len(terms)} unique medical terms.")
    
    term_embeddings = {}
    print("Generating BioBERT embeddings for terms...")
    for term in terms:
        inputs = tokenizer(term, return_tensors="pt", truncation=True, padding=True)
        with torch.no_grad():
            outputs = biobert_model(**inputs)
            # Use the CLS token embedding (first token) as the term embedding
            embedding = outputs.last_hidden_state[:, 0, :].squeeze().numpy()
            term_embeddings[term] = embedding
            
    return terms, term_embeddings

# Example usage:
# terms, embeddings = extract_and_embed_medical_terms(raw_text)
# print("Extracted terms:", terms)

Loading BioBERT model and tokenizer...
Loading NER Pipeline...


Device set to use cpu


## 4. LLM Simplification and General Advice
Pass the raw OCR text and extracted terms to the Generative AI (Gemini) over API to produce a simple, structured summary.

In [6]:
def simplify_medical_report(raw_text, medical_terms):
    if not api_key:
        return "Error: API key not configured. Cannot call LLM."

    prompt = f"""
You are a helpful, empathetic medical assistant. Your job is to simplify a medical report for a patient who has no medical background.

Here is the raw text extracted from the patient's medical report:
{raw_text}

Key medical terms we identified in the report:
{', '.join(medical_terms) if medical_terms else 'None specifically identified'}

Please provide a response with the following sections:
1. **Summary**: A brief, easy-to-understand summary of the report.
2. **Key Findings (What are the problems?)**: Explain the abnormal results or main findings in simple terms. Avoid complex jargon.
3. **Explanation of Medical Terms**: Briefly define the key medical terms found in the report. (Use the list above to guide you, but prioritize terms that are crucial for understanding the report).
4. **General Advice**: Provide general, non-diagnostic advice (e.g., "Consult your doctor for a detailed diagnosis", "Stay hydrated", etc.).

IMPORTANT: Maintain a supportive and objective tone. Add a disclaimer that you are an AI assistant and this is not a substitute for professional medical advice.
"""

    print("Sending request to LLM...")
    response = model.generate_content(prompt)
    return response.text

# Example usage:
# simplified_report = simplify_medical_report(raw_text, terms)
# print(simplified_report)

## 5. Main Execution Pipeline
Connects all the dots into a single pipeline.

In [7]:
def process_medical_report(file_path):
    print(f"--- Processing {file_path} ---")
    try:
        # 1. OCR
        raw_text = extract_text(file_path)
        print("✅ Text successfully extracted.")
        
        # 2. Term Extraction & Embedding
        # For development, you might want to skip or comment out large text blocks to avoid limits
        terms, embeddings = extract_and_embed_medical_terms(raw_text)
        print(f"✅ Extracted {len(terms)} medical terms and generated their embeddings.")
        
        # 3. LLM Simplification
        simplified_output = simplify_medical_report(raw_text, terms)
        print("✅ Simplified report generated.\n")
        
        print("==================================================")
        print("🩺 SIMPLIFIED MEDICAL REPORT")
        print("==================================================")
        print(simplified_output)
        
    except Exception as e:
        print(f"An error occurred: {e}")

# Instructions to user:
# 1. Upload your medical report to the same folder as this notebook.
# 2. Ensure your .env file is set up with GEMINI_API_KEY.
# 3. Uncomment the line below and replace 'your_report.pdf' with the actual file name.

process_medical_report("2f8280772fb0d5f9ec70e1ba0866b99f.jpg")

--- Processing 2f8280772fb0d5f9ec70e1ba0866b99f.jpg ---
✅ Text successfully extracted.
Extracting medical terms from text...
Found 31 unique medical terms.
Generating BioBERT embeddings for terms...
✅ Extracted 31 medical terms and generated their embeddings.
Sending request to LLM...
✅ Simplified report generated.

🩺 SIMPLIFIED MEDICAL REPORT
Hello Yashvi, thank you for trusting me to help you understand your medical report. I know medical reports can sometimes be confusing, but I'll do my best to explain it clearly and empathetically.

---

### 1. Summary

This report is about a blood test called **B-type Natriuretic Peptide (BNP)**. This test helps doctors check on your heart health.

The good news is that your BNP level is **18.00 pg/mL**, which is **within the normal range** for this test (the normal range is typically less than 29.40 pg/mL). This means that, based on this specific test, your heart does not appear to be under the kind of stress that would cause elevated BNP levels